In [ ]:
#| hide
from hello_claude.core import *

# Hello World: nbdev + Claude Code on GitHub

> 

A minimal end-to-end walkthrough for building a Python package with notebook-driven development (**nbdev**), wired up to **Claude Code** so that GitHub issues can be triaged and fixed automatically via `@claude` mentions.

You write notebooks. nbdev exports modules, runs tests, and publishes docs. Claude handles small fixes, refactors, and keeps docs in sync. You stay in the loop and review every PR.

---

## What you'll have at the end



- A public GitHub repo with a working Python package
- Auto-generated documentation site on GitHub Pages
- CI that tests notebooks on every push
- A `@claude` bot that responds to issues and PR comments, opens branches, edits the right notebook, runs `nbdev-prepare`, and submits PRs

---

## Prerequisites

- macOS, Linux, or Windows under WSL (nbdev does **not** support plain cmd/PowerShell)
- Python 3.10+ with pip
- A GitHub account
- Either an **Anthropic API key** (from `console.anthropic.com`) **or** a **Claude Pro/Max subscription**
- Optional but helpful: the [GitHub CLI (`gh`)](https://cli.github.com/)

---

## Part 1 — Install the tooling

You only do this once per machine.

```bash
# JupyterLab
pip install jupyterlab

# nbdev itself
pip install nbdev

# Quarto (the docs engine; will ask for sudo)
nbdev-install-quarto

# Quarto's JupyterLab extension (nicer rendering while you edit)
pip install jupyterlab-quarto

# Claude Code CLI
npm install -g @anthropic-ai/claude-code

Verify:

```bash
nbdev-help     # should print a list of nbdev-* commands
claude --version
```

---

## Part 2 — Create an empty GitHub repo

Go to [github.com/new](https://github.com/new) and create a new repo. Use the name `hello-claude` (or whatever you prefer).

> ⚠️ **Important:** Add a description (nbdev will use it later), but do **not** add a README, `.gitignore`, or license at this point. nbdev will generate those for you.

You should land on an empty repo page after clicking *Create repository*.

---

## Part 3 — Initialise the repo with nbdev

Clone the empty repo and `cd` into it:

```bash
git clone https://github.com/<your-user>/hello-claude.git
cd hello-claude
```

You'll see the warning *"You appear to have cloned an empty repository"* — that's expected. Now scaffold the project:

```bash
nbdev-new
```

- `pyproject.toml` (project metadata — nbdev3 uses this instead of the old `settings.ini`)
- `nbs/00_core.ipynb` and `nbs/index.ipynb` (your starter notebooks)
- `hello_claude/` (the Python package, name derived from the repo name with `-` replaced by `_`)
- `.github/workflows/test.yaml` (CI: notebook tests + sync check)
- `.github/workflows/deploy.yaml` (builds and publishes docs to GitHub Pages)
- `LICENSE`, `.gitignore`, `MANIFEST.in`

Quick sanity check on `pyproject.toml`, then push:

```bash
git add .
git commit -m "Initial nbdev scaffold"
git push
```

---

## Part 4 — Enable GitHub Pages

In your repo on GitHub: **Settings → Pages → Branch: `gh-pages` → Save**.

If `gh-pages` isn't in the dropdown yet, wait a minute for the first deploy workflow to create it, then refresh.

Open the **Actions** tab. You should see two workflow runs: *CI* and *Deploy to GitHub Pages*. After they finish, your docs are live at `https://<your-user>.github.io/hello-claude`.

---

## Part 5 — Make your first edit

Install nbdev's git hooks so notebooks stay diff-friendly:

```bash
nbdev-install-hooks
```

Install your package in editable mode (so you can `import hello_claude` from anywhere):

```bash
pip install -e '.[dev]'
```

Start a live docs preview in a side terminal — you'll see changes as you save notebooks:

```bash
nbdev-preview
```

Now open Jupyter (`jupyter lab`) and edit `nbs/00_core.ipynb`. Replace the placeholder function with:

```python
#| export
def say_hello(to: str) -> str:
    "Say hello to somebody"
    return f"Hello {to}!"
```

Add a test cell below it:

```python
assert say_hello("Claude") == "Hello Claude!"
```

Restart the kernel and run all cells (the ⏩ button). Then in the terminal:

```bash
nbdev-prepare
```

This runs `nbdev-export` (notebooks → `.py`), `nbdev-test` (executes all notebook cells), `nbdev-clean` (strips junk metadata), and `nbdev-readme` (regenerates `README.md` from `index.ipynb`).

Commit and push:

```bash
git add .
git commit -m "feat: add say_hello"
git push
```

Check the Actions tab — CI should pass green ✅, and the docs site updates within a minute.

---

## Part 6 — Wire up Claude Code

This is where the magic kicks in.

### 6a. Pick an authentication method

| Option | When to use | Cost model |
|---|---|---|
| **API key** | Team repos, predictable billing, heavy automation | Pay per token (typically <$5/mo for small teams) |
| **OAuth token (Pro/Max)** | Personal projects, you already have a subscription | Uses your subscription quota — shared with interactive use |

> ⚠️ If both `ANTHROPIC_API_KEY` and `CLAUDE_CODE_OAUTH_TOKEN` are set in the repo, the **API key wins** and the subscription is bypassed.

### 6b. Install the GitHub App

From the repo root:

```bash
claude
```

Then inside Claude Code:

```
/install-github-app
```

Follow the wizard. It will:

1. Open a browser to install the Anthropic GitHub App on this repo
2. Ask which auth method you want
3. For OAuth: prompt you to run `claude setup-token` and paste the result
4. Commit `.github/workflows/claude.yml` to a new branch and open a PR

Merge that PR. From this point on, `@claude` mentions in issues and PR comments will trigger the action.

### 6c. The generated workflow (for reference)

```yaml
name: Claude Code
on:
  issue_comment:
    types: [created]
  pull_request_review_comment:
    types: [created]
  issues:
    types: [opened, assigned]
  pull_request_review:
    types: [submitted]

jobs:
  claude:
    if: contains(github.event.comment.body, '@claude') || contains(github.event.issue.body, '@claude')
    runs-on: ubuntu-latest
    permissions:
      contents: write
      pull-requests: write
      issues: write
      id-token: write
    steps:
      - uses: actions/checkout@v4
      - uses: anthropics/claude-code-action@v1
        with:
          claude_code_oauth_token: ${{ secrets.CLAUDE_CODE_OAUTH_TOKEN }}
          # OR for API-key auth:
          # anthropic_api_key: ${{ secrets.ANTHROPIC_API_KEY }}
```

---

## Part 7 — Tell Claude how nbdev works

This is the **single most important step** for this stack. Without it, Claude will edit the auto-generated `.py` files in `hello_claude/`, and your next `nbdev-export` will silently overwrite its work.

Create `CLAUDE.md` in the repo root:

```markdown
# Project Conventions

This is an **nbdev3** project. **Notebooks in `nbs/` are the source of truth.**

## Rules

- **Never edit `.py` files in `hello_claude/` directly** — they are auto-generated.
- All code changes happen in `nbs/*.ipynb` notebooks.
- After editing a notebook, run `nbdev-prepare` to sync modules, run tests,
  clean notebooks, and update `README.md`.
- New public functions need:
  - A docstring on the first line
  - The `#| export` directive at the top of the cell
  - At least one test cell with `assert` statements directly below
- Use `#| hide` for cells that shouldn't appear in docs.
- Commit messages follow Conventional Commits: `feat:`, `fix:`, `docs:`, `test:`, `refactor:`.

## Workflow for fixing an issue

1. Read the issue carefully.
2. Identify the notebook in `nbs/` that owns the relevant module.
3. Edit the notebook (not the `.py` file).
4. Add or update test cells in the same notebook.
5. Run `nbdev-prepare` and confirm tests pass.
6. Commit on a new branch named `claude/<issue-number>-<short-slug>`.
7. Open a PR referencing the issue with `Fixes #<number>`.

## Useful commands

- `nbdev-export` — notebooks → .py modules
- `nbdev-test` — run all notebook-based tests
- `nbdev-clean` — strip metadata from notebooks
- `nbdev-prepare` — all of the above plus README sync
- `nbdev-preview` — local docs preview (not needed in CI)
```

Commit and push:

```bash
git add CLAUDE.md
git commit -m "docs: add CLAUDE.md with nbdev conventions"
git push
```

---

## Part 8 — Smoke test

Open a new issue on GitHub:

> **Title:** Add German greeting
>
> **Body:**
> `@claude` please add a `sag_hallo(name)` function to `nbs/00_core.ipynb` that returns `"Hallo {name}!"`. Include a test cell with an `assert`, run `nbdev-prepare` before committing, and open a PR.

What happens next:

1. **Actions tab** — the *Claude Code* workflow starts within seconds
2. Claude reads the issue, `CLAUDE.md`, and the notebook
3. It creates a branch like `claude/1-add-german-greeting`
4. It edits `nbs/00_core.ipynb`, runs `nbdev-prepare`
5. It opens a PR that references the issue
6. nbdev's own `test.yaml` workflow runs against the PR
7. You review, request changes if needed (via `@claude` in PR comments), and merge
8. The `deploy.yaml` workflow rebuilds and publishes the docs site

---

## Common pitfalls

| Symptom | Cause | Fix |
|---|---|---|
| Claude edited `.py` files instead of `.ipynb` | `CLAUDE.md` not clear enough, or not present | Strengthen `CLAUDE.md`, add an explicit "never touch X" rule, mention `nbdev-export` |
| CI fails with "notebooks and modules out of sync" | Claude or you forgot `nbdev-prepare` before commit | `nbdev-prepare && git add . && git commit --amend` |
| Quarto-related errors in CI | Workflow runner is missing Quarto | nbdev's default `test.yaml` handles this; in custom workflows call `nbdev-install-quarto` |
| Notebook merge conflicts | Hooks not installed | `nbdev-install-hooks` (per repo) |
| OAuth subscription quota burns fast | Heavy automation triggers (`pull_request: [opened, synchronize]`) | Switch to API-key auth, or narrow trigger conditions |
| `@claude` mention does nothing | App not installed on this repo, or PR with workflow not merged yet | Re-run `/install-github-app`, check repo's installed GitHub Apps |

---

## What this stack gives you

- **You stay in control of design.** Claude only touches what you ask via issues/PR comments.
- **Documentation is never stale.** Notebooks are simultaneously source, tests, and docs.
- **Releases are one command.** `nbdev-pypi` or `nbdev-release-both` publishes to PyPI/conda when you're ready.
- **Two safety nets.** Claude's PR has to pass nbdev's CI before you merge.

---

## Resources

- [nbdev documentation](https://nbdev.fast.ai/)
- [nbdev end-to-end tutorial](https://nbdev.fast.ai/tutorials/tutorial.html)
- [Claude Code documentation](https://docs.claude.com/en/docs/claude-code/overview)
- [Claude Code GitHub Actions guide](https://docs.claude.com/en/docs/claude-code/github-actions)
- [`anthropics/claude-code-action` repository](https://github.com/anthropics/claude-code-action)